# Scam Call Shield — detector training on AMD GPU (ROCm)

Trains both HUMAN-vs-AI voice detectors on an **AMD Instinct GPU** via ROCm PyTorch:
1. CNN baseline (`voice_classifier/train.py`) — kept as the honest failed baseline
2. **wav2vec2-base fine-tune** (`voice_classifier/finetune_w2v.py`) — the production model

ROCm PyTorch exposes the GPU through the standard `torch.cuda` API, so the code runs unmodified.
The training logs written to `voice_trends/` record the GPU device name — commit them as AMD-usage evidence.

> **Important:** every shell cell below uses `{sys.executable}` (the kernel's own
> Python), NOT a bare `!python`. A bare `!python` in Jupyter can resolve to a
> different interpreter with a CPU-only torch — which makes training silently run
> on the CPU even though this kernel sees the AMD GPU. `sys.executable` guarantees
> the subprocess is the same ROCm Python as cell 1.

**Two run sizes:**
- **Minimal qualifying run** — steps 1–4: LibriSpeech + edge-tts only (~2 GB, ~30 min).
- **Full run** — adds step 5 (In-the-Wild + XTTS-v2 clones, +9 GB) and reproduces the
  README's 4-tier table. Reference numbers from the RTX 4090 run: 100% unseen-TTS /
  84.6% unseen-speaker XTTS clones / 98.6% In-the-Wild deepfakes @ 6.9% FPR;
  YouTube fake-Trump 0.791, real Trump 0.008.

In [ ]:
# 1. Environment check — should print an AMD GPU (e.g. 'AMD Instinct MI300X') and is ROCm: True
import sys, torch
print('python exe :', sys.executable)   # every shell cell below reuses THIS interpreter
print('torch      :', torch.__version__)
print('gpu ok     :', torch.cuda.is_available())
print('device     :', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('is ROCm    :', torch.version.hip is not None, '| hip:', torch.version.hip)
assert torch.cuda.is_available(), 'GPU not visible to this kernel — fix before training'

In [ ]:
# If torch is missing or CPU-only on this AMD machine, install ROCm wheels FIRST, then restart the kernel:
# %pip install torch torchaudio --index-url https://download.pytorch.org/whl/rocm6.2
%pip install -q soundfile librosa edge-tts tqdm scikit-learn transformers

In [ ]:
# 2. Get the code (skip if you uploaded the repo manually)
# !git clone https://github.com/Vishalkagade/amd-hackthaon.git
# %cd amd-hackthaon
import os
print('working dir:', os.getcwd())
assert os.path.exists('voice_classifier/train.py'), 'run this notebook from the repo root'

In [ ]:
# 3. Build the base dataset: LibriSpeech (human) + edge-tts neural voices (AI)
#    ~15-30 min depending on network. Idempotent — safe to re-run.
!{sys.executable} -m voice_classifier.prepare_data --max-human 1200 --max-ai 1200

In [ ]:
# 4. Train BOTH detectors on the AMD GPU. Checkpoints + logs land in voice_trends/
#    The training header MUST print 'device=cuda (AMD Instinct ...)'. If it says
#    device=cpu, the kernel's torch is CPU-only — reinstall ROCm wheels (cell 2).
!{sys.executable} -m voice_classifier.train --epochs 14 --batch-size 64 --out voice_trends
!{sys.executable} -m voice_classifier.finetune_w2v --epochs 5

In [ ]:
# 5. OPTIONAL full pipeline — real-world deepfakes + clone attack sets (+~9 GB).
#    Uncomment to run, then re-run step 4 so training picks the new data up.

#    5a. In-the-Wild dataset (31.8k real internet deepfakes + genuine celebrity audio):
# !mkdir -p data_raw && curl -sL -o data_raw/itw.zip 'https://owncloud.fraunhofer.de/index.php/s/JZgXh0JEAF0elxa/download'
# !cd data_raw && unzip -q itw.zip && rm itw.zip

#    5b. XTTS-v2 clones — Coqui TTS needs a SEPARATE venv (pins transformers 4.x / torch 2.8).
#        This one intentionally uses .venv-tts/bin/python, NOT sys.executable.
# !python3 -m venv .venv-tts && .venv-tts/bin/pip install -q 'coqui-tts[codec]' 'transformers<5' 'torch==2.8.*' 'torchaudio==2.8.*' librosa soundfile tqdm
# !COQUI_TOS_AGREED=1 .venv-tts/bin/python -m voice_classifier.make_xtts_clones --n 350
# !COQUI_TOS_AGREED=1 .venv-tts/bin/python -m voice_classifier.make_xtts_clones --itw-refs --n 120

#    5c. Re-train with everything, then the 4-tier evaluation:
# !{sys.executable} -m voice_classifier.train --epochs 14 --out voice_trends
# !{sys.executable} -m voice_classifier.finetune_w2v --epochs 5
# !{sys.executable} -m voice_classifier.eval_deepfake

In [ ]:
# 6. AMD-usage evidence: the logs record which GPU trained the models. COMMIT THESE FILES.
import json
for log_path in ('voice_trends/training_log.json',
                 'voice_trends/wav2vec2_finetuned/finetune_log.json'):
    try:
        log = json.load(open(log_path))
        print(f"{log_path}\n  trained on: {log['gpu']}  torch: {log['torch']}"
              f"  rocm: {log.get('rocm', 'n/a')}  best: {log['best_val_acc']:.4f}\n")
    except FileNotFoundError:
        print(f'{log_path}: missing — run step 4 first\n')

In [ ]:
# 7. Sanity inference with the production detector: dataset samples...
from voice_classifier.infer import load_best_detector
from pathlib import Path
det, name = load_best_detector()
print('using:', name)
h = next(Path('data/human').glob('*.wav')); a = next(Path('data/ai').glob('*.wav'))
print('human clip -> ai_prob =', round(det.score_file(str(h))[0], 3))
print('ai clip    -> ai_prob =', round(det.score_file(str(a))[0], 3))

In [ ]:
# 8. ...and the hard samples committed in the repo (never in any training/val set).
#    Expected (full run): real Trump/Obama LOW (~0.01), fake_trump ~0.79,
#    mms ~0.99. Minimal run (steps 1-4 only) will miss fake_trump — that gap is
#    exactly what step 5's clone/ITW hardening fixes.
from pathlib import Path
for f in sorted(Path('hard_samples').glob('*.wav')):
    p, per = det.score_file(str(f))
    print(f'{f.name:25s} ai_prob={p:.3f}  windows>0.5: {sum(1 for x in per if x > 0.5)}/{len(per)}')